<a href="https://colab.research.google.com/github/emmanuelle-bul/AI_Exercises/blob/main/FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

In [ ]:
import random
import json

from datasets import load_dataset

In [ ]:
# dataset from https://huggingface.co/datasets/
retail_banking_dataset = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset")

README.md:   0%|          | 0.00/11.7k [00:00<?, ?B/s]

bitext-retail-banking-llm-chatbot-traini(…): reconstructing file:   0%|          |  0.00B / 7.87MB            

bitext-retail-banking-llm-chatbot-traini(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25545 [00:00<?, ? examples/s]

In [ ]:
for split_name, split_data in retail_banking_dataset.items():
  split_data.to_csv(f"/content/dataset_{split_name}.csv")
  split_data.to_json(f"/content/dataset_{split_name}.jsonl")

  random_indices = random.sample(range(split_data.num_rows), k=10) # вземаме случайни 20 реда измежду всички
  sample_data = split_data.select(random_indices)   #извлича от dataset-a конкретните случайно селектирани индекси

  sample_data.to_csv(f"/content/sample_{split_name}.csv")

  random_indices = random.sample(range(split_data.num_rows), k=600)
  # FineTuning
  fine_tuting_data = split_data.select(random_indices).map(
      lambda x: {
          "messages": [
              { "role": "system", "content": "You are a helpful customer support expert for banking/financial services.\nResponse style and safety:\nReturn valid JSON only, with exactly the keys: category, intent, response."},
              { "role": "user", "content": f"User Request:\n{x['instruction']}"},
              { "role": "assitent", "content": json.dumps({ "category": x['category'], "intent": x['intent'], "response": x['response'] }) }
          ]
      },
      remove_columns=split_data.column_names  # премахни колоните от оригиналния dataset
  )

  print(fine_tuting_data)

  fine_tuting_data.to_json(f"/content/fine_tuning_{split_name}.jsonl")

Creating CSV from Arrow format:   0%|          | 0/26 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/26 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 600
})


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]